# 🔬 Model Explainability (SHAP & LIME)
**Extended · Data Pattern Recognition for the Rest of Us**

> A black-box model that performs well is only half the job. Regulators, customers, and business leaders need to understand *why* a prediction was made. SHAP and LIME make any model explainable.

### Dataset
**Loan Approval / Credit Risk**
Predict loan default using applicant features: income, employment, credit history, debt ratio. This is one of the highest-stakes explainability domains — regulations (ECOA, FCRA) often require lenders to provide specific reasons for adverse decisions.

### Time: ~65 minutes

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa',
    'axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
!pip install shap -q
import shap
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report
from sklearn.preprocessing import StandardScaler
print("\u2713 Setup complete — shap installed")

In [ ]:
# Loan Application Dataset
# Predict probability of default — a regulated, high-stakes prediction
np.random.seed(42)
n = 5000

# Applicant features
credit_score     = np.random.randint(300, 850, n)
annual_income    = np.random.lognormal(11, 0.5, n).astype(int).clip(15000, 500000)
employment_years = np.random.randint(0, 30, n).astype(float)
debt_to_income   = np.random.beta(2, 5, n).clip(0.01, 0.99)
loan_amount      = np.random.lognormal(10.5, 0.5, n).astype(int).clip(1000, 100000)
num_credit_lines = np.random.randint(1, 20, n)
late_payments    = np.random.choice([0,1,2,3,4,5], n, p=[0.60,0.18,0.10,0.06,0.04,0.02])
home_ownership   = np.random.choice(['Rent','Own','Mortgage'], n, p=[0.35,0.20,0.45])
loan_purpose     = np.random.choice(['Debt Consolidation','Home Improvement',
                                     'Auto','Business','Education'], n,
                                    p=[0.40,0.20,0.15,0.15,0.10])

# Default probability — driven by credit score, DTI, late payments
log_odds = (-1
            - 0.008 * (credit_score - 600)
            + 2.5   * debt_to_income
            + 0.4   * late_payments
            - 0.0000015 * annual_income
            - 0.03  * employment_years
            + 0.000003 * loan_amount)
prob_default = 1 / (1 + np.exp(-log_odds))
default = (np.random.uniform(0,1,n) < prob_default).astype(int)

df = pd.DataFrame({
    'CreditScore': credit_score, 'AnnualIncome': annual_income,
    'EmploymentYears': employment_years, 'DebtToIncome': debt_to_income,
    'LoanAmount': loan_amount, 'NumCreditLines': num_credit_lines,
    'LatePayments': late_payments,
    'HomeOwnership': pd.Categorical(home_ownership).codes,
    'LoanPurpose': pd.Categorical(loan_purpose).codes,
    'Default': default
})

feature_cols = [c for c in df.columns if c != 'Default']
X = df[feature_cols]
y = df['Default']
print(f"Loan dataset: {n:,} applications")
print(f"Default rate: {y.mean():.1%}")
print(f"\nFeatures: {feature_cols}")

## 📐 Part 1 — SHAP Values: Explaining Every Prediction

**SHAP** (SHapley Additive exPlanations) assigns each feature a contribution to the difference between this specific prediction and the average prediction.

```
prediction = base_value + SHAP(CreditScore) + SHAP(DTI) + SHAP(LatePayments) + ...
```

Key properties:
- **Efficiency:** SHAP values sum exactly to the prediction deviation from baseline
- **Consistency:** features that contribute more always get higher SHAP values
- **Local + Global:** explain one prediction OR summarize across all predictions

In [ ]:
# Fit a GBM model on loan data
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                             random_state=42, stratify=y)
gbm = GradientBoostingClassifier(n_estimators=200, max_depth=4,
                                  learning_rate=0.05, random_state=42)
gbm.fit(X_tr, y_tr)
print(f"GBM AUC-ROC: {roc_auc_score(y_te, gbm.predict_proba(X_te)[:,1]):.4f}")
print("\nCalculating SHAP values (may take ~30 seconds)...")

explainer = shap.TreeExplainer(gbm)
shap_values = explainer.shap_values(X_te)
print("\u2713 SHAP values computed")

In [ ]:
# Global: SHAP summary plot — what drives default risk overall?
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_te, plot_type='bar', show=False,
                  plot_size=(9, 5))
plt.title('Global Feature Importance — What Drives Loan Default Risk?\n'
          'Average absolute SHAP value across all predictions')
plt.tight_layout(); plt.show()

In [ ]:
# SHAP beeswarm — direction AND magnitude
plt.figure(figsize=(9, 6))
shap.summary_plot(shap_values, X_te, show=False, plot_size=(9, 6))
plt.title('SHAP Beeswarm Plot — Direction and Magnitude\n'
          'Red = high feature value, Blue = low feature value')
plt.tight_layout(); plt.show()
print("\n\u2714 Reading this plot:")
print("  CreditScore: high score (red) = negative SHAP = reduces default risk")
print("  DebtToIncome: high DTI (red) = positive SHAP = increases default risk")
print("  LatePayments: any late payments push default risk up significantly")

In [ ]:
# Local explanation: explain ONE specific loan application
# Regulatory requirement: adverse action notice must give specific reasons
sample_idx = np.where(y_te.values == 1)[0][0]  # find a defaulted loan
applicant = X_te.iloc[[sample_idx]]

pred_prob = gbm.predict_proba(applicant)[0,1]
shap_for_applicant = shap_values[sample_idx]

print("=== Adverse Action Notice — Loan Application ===\n")
print(f"Applicant Profile:")
for feat, val in zip(feature_cols, applicant.values[0]):
    print(f"  {feat:<20} {val:>12.2f}")
print(f"\nPredicted default probability: {pred_prob:.1%}")
print(f"Decision: {'DECLINE' if pred_prob > 0.5 else 'APPROVE'}\n")

# Top reasons (highest positive SHAP = most contributed to default)
shap_df = pd.DataFrame({'Feature': feature_cols,
                         'SHAP': shap_for_applicant,
                         'Value': applicant.values[0]})
shap_df = shap_df.sort_values('SHAP', ascending=False)

print("Reasons for decision (required by ECOA/FCRA):")
for _, row in shap_df.head(4).iterrows():
    direction = 'INCREASES' if row['SHAP'] > 0 else 'REDUCES'
    print(f"  {row['Feature']:<22} {direction} default risk  (SHAP={row['SHAP']:+.3f})")

fig, ax = plt.subplots(figsize=(10, 4))
shap.waterfall_plot(shap.Explanation(
    values=shap_for_applicant,
    base_values=explainer.expected_value,
    data=applicant.values[0],
    feature_names=feature_cols), show=False)
plt.title(f'Waterfall Plot — Why This Application Was {"Declined" if pred_prob>0.5 else "Approved"}\n'
          f'Predicted default probability: {pred_prob:.1%}')
plt.tight_layout(); plt.show()

In [ ]:
# Stakeholder memo template
top_risks = shap_df[shap_df['SHAP']>0].head(3)
print("\n=== STAKEHOLDER MEMO (plain English explanation) ===")
print(f"\nApplication Decision: {'DECLINED' if pred_prob>0.5 else 'APPROVED'}")
print(f"Default Probability: {pred_prob:.0%}\n")
print("The three primary factors contributing to this decision are:")
for i, (_, row) in enumerate(top_risks.iterrows(), 1):
    print(f"  {i}. {row['Feature']}: current value of {row['Value']:.2f} "
          f"increases default risk (contribution score: {row['SHAP']:+.3f})")
print("\nNote: These factors are ranked by their individual contribution to the")
print("prediction, holding all other factors constant. Improving any of these")
print("factors would improve the applicant's credit profile.")

In [ ]:
answers = {
    "q1": "",  # What does a SHAP value of +0.25 for DebtToIncome mean?
    "q2": "",  # How is SHAP different from standard feature importance (like Gini importance)?
    "q3": "",  # What regulatory requirement makes explainability essential for loan decisions?
    "q4": "",  # What is LIME and how does it differ from SHAP?
    "q5": "",  # Write a 2-sentence plain English explanation of a loan decline for a non-technical applicant.
}
missing=[k for k,v in answers.items() if not str(v).strip()]
print("\u274c Still empty: "+str(missing) if missing else "\u2705 Done! File \u2192 Save a copy in GitHub")

---
*Data Pattern Recognition for the Rest of Us · [ladataanalytics.github.io/pattern-recognition-notebooks](https://ladataanalytics.github.io/pattern-recognition-notebooks)*